## TECH CHALLENGE - FASE 4

## Análise Multimodal para Detecção de Riscos em Saúde da Mulher

Este notebook apresenta a implementação inicial de um pipeline multimodal capaz de analisar dados de vídeo e áudio com o objetivo de identificar possíveis sinais de risco clínico e psicológico.

A solução é uma evolução do assistente médico desenvolvido na fase anterior, incorporando análise de comportamento (vídeo) e linguagem (áudio), permitindo maior capacidade de detecção precoce de situações críticas.

## Objetivo

Desenvolver um pipeline multimodal que:

- Analise vídeos para identificar padrões comportamentais e sinais de desconforto
- Processe áudios de consultas para detectar sinais emocionais e psicológicos
- Combine os resultados das diferentes modalidades
- Gere um alerta baseado no nível de risco identificado

Este sistema atua como suporte à triagem clínica, não realizando diagnósticos médicos.

## Importação de Bibliotecas e Módulos

Nesta etapa, são importadas as funções desenvolvidas no projeto para análise de vídeo, áudio e fusão multimodal.

In [1]:
import sys
sys.path.append("..")

In [2]:
from src.multimodal.video_processor import analyze_video
from src.multimodal.audio_processor import analyze_audio
from src.multimodal.multimodal_fusion import calculate_multimodal_risk
from src.multimodal.alert_generator import generate_alert

## Configuração do Ambiente

Configuração de caminhos e preparação do ambiente para execução das análises.

In [3]:
from pathlib import Path

print(Path.cwd())

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks


In [4]:
video_result = analyze_video("../data/multimodal/video/sample.mp4")
audio_result = analyze_audio("../data/multimodal/audio/sample.wav")

risk_result = calculate_multimodal_risk(video_result, audio_result)
alert = generate_alert(risk_result)

video_result, audio_result, risk_result, alert

({'modality': 'video',
  'risk_score': 0,
  'risk_level': 'not_provided',
  'flags': ['video_not_opened'],
  'metadata': {'video_path': '..\\data\\multimodal\\video\\sample.mp4',
   'fps': 0,
   'frame_count': 0,
   'duration_seconds': 0,
   'status': 'mock_video_file'}},
 {'modality': 'audio',
  'risk_score': 0.25,
  'risk_level': 'baixo',
  'flags': [],
  'detected_categories': {},
  'voice_features': {'duration_seconds': None,
   'rms_energy': None,
   'voice_intensity': 'indisponível'},
  'transcription': 'Não foi possível transcrever o áudio.',
  'interpretation': 'Não foram identificados sinais emocionais relevantes na transcrição. A recomendação é manter acompanhamento regular.',
  'recommendation': 'Recomenda-se acompanhamento regular.'},
 {'final_score': 0.15,
  'risk_level': 'baixo',
  'alert': False,
  'evidences': ['video_not_opened'],
  'video_score': 0,
  'audio_score': 0.25},
 'Sem alerta crítico no momento. Recomenda-se acompanhamento regular pela equipe de saúde.')

## Carregamento de Dados de Entrada

Nesta fase inicial, são utilizados dados simulados (mock) para representar arquivos de vídeo e áudio.
Em etapas futuras, estes dados serão substituídos por arquivos reais e integrados a serviços como Azure Speech e modelos de visão computacional.

## Análise de Vídeo

O módulo de análise de vídeo tem como objetivo identificar padrões visuais que possam indicar:

- Desconforto físico
- Movimentos anormais
- Postura corporal inadequada

Nesta etapa inicial, a análise é simulada e será futuramente substituída por modelos de visão computacional (YOLOv8).

## Análise de Áudio

O módulo de análise de áudio processa informações da fala da paciente, buscando identificar:

- Sinais de ansiedade
- Indícios de medo
- Relatos de sintomas relevantes

A análise utiliza transcrição simulada, sendo posteriormente integrada com serviços como Azure Speech-to-Text.

In [4]:
import sys
sys.path.append("..")

from src.workflows.langgraph_flow import build_medical_assistant_graph
from src.llm.ollama_client import get_llm
from src.rag.vector_store import load_vector_store
from src.rag.retriever import get_retriever
from src.rag.retriever import retrieve_context

In [5]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks\..\src\llm\ollama_client.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


Base vetorial carregada de: ../data/vector_store


## Execução do Fluxo Multimodal com LangGraph

Nesta etapa, o pipeline multimodal é executado por meio do LangGraph, mantendo a arquitetura evolutiva desenvolvida na fase anterior.

O fluxo recebe como entrada uma pergunta clínica, um identificador opcional de paciente, um arquivo de vídeo e um arquivo de áudio. Em seguida, o sistema executa os nós de guardrails, recuperação de contexto via RAG, análise de vídeo, análise de áudio, fusão multimodal, geração de resposta e registro da interação.

Nesta execução inicial, os dados de áudio e vídeo ainda são simulados, permitindo validar a integração entre os módulos antes da substituição por modelos reais, como YOLOv8 para vídeo, Azure Speech-to-Text para transcrição e GPT para interpretação textual especializada.

In [7]:
state = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/sample.wav"
}

result = app.invoke(state)
result

{'question': 'Avalie possíveis sinais de risco no atendimento da paciente.',
 'risk_level': 'low',
 'action': 'allow',
 'reason': 'Pergunta informativa.',
 'docs': [Document(id='b0801d89-44f4-4f6c-8a1e-b74b7a03c78a', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='fe392124-22dc-406d-a1cb-da95babe86e9', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='a29fa53d-b57a-47d2-a544-780df6879c83', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia')],
 'context': 'Recurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia',
 'patient_id': None,
 'patient_context': 'No patien

## Interpretação do Resultado

O resultado demonstra que o fluxo multimodal foi executado com sucesso.

A saída contém:

- A classificação de risco inicial realizada pelos guardrails
- Os documentos recuperados pela base vetorial FAISS
- O contexto estruturado do paciente, quando disponível
- O contexto multimodal gerado a partir das análises de vídeo e áudio
- O score final de risco multimodal
- A resposta gerada pelo assistente médico
- As fontes utilizadas para rastreabilidade
- O status final da execução

Nesta primeira versão, o risco final foi classificado como médio, com evidências simuladas relacionadas a movimento corporal, medo, cansaço e dificuldade para dormir. O alerta crítico não foi acionado, mas o sistema recomenda acompanhamento regular pela equipe de saúde.

Essa etapa confirma que a arquitetura da Fase 3 foi expandida para suportar dados multimodais, mantendo os princípios de segurança, explicabilidade e rastreabilidade.

In [8]:
print(result["answer"])
print(result["multimodal_result"])

 Based on the provided structured patient data, multimodal analysis, and evidence, there are no critical alerts detected in this patient's current assessment. The multimodal score is 0.4, indicating a medium risk level. The patient has shown identified body movement and an initial visual analysis has been executed. Therefore, it is recommended to continue regular follow-up with the healthcare team.

However, it's important to note that Recurrent Adult Acute Lymphoblastic Leukemia (ALL) was mentioned in the medical knowledge retrieved section. This information might be relevant to the patient's condition, but without specific patient data, it is not possible to definitively assess its significance. It would be advisable for the healthcare team to consider this potential diagnosis during further evaluation and follow-up.

In summary, while there are no critical alerts at the moment, the healthcare team should remain vigilant regarding the possibility of Recurrent Adult Acute Lymphoblasti

In [9]:
print(result["multimodal_context"])

MULTIMODAL ANALYSIS:
Video risk score: 0.55
Audio risk score: 0.3
Final multimodal score: 0.4
Risk level: medio
Alert generated: False

Evidence:
- movimento corporal identificado
- análise visual inicial executada

Alert message:
Sem alerta crítico no momento. Recomenda-se acompanhamento regular pela equipe de saúde.


In [10]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app1 = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

Base vetorial carregada de: ../data/vector_store


In [11]:
state = {
    "question": "Avalie possíveis sinais de câncer de mama.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/sample.wav"
}

result1 = app1.invoke(state)
result1

{'question': 'Avalie possíveis sinais de câncer de mama.',
 'risk_level': 'low',
 'action': 'allow',
 'reason': 'Pergunta informativa.',
 'docs': [Document(id='b0801d89-44f4-4f6c-8a1e-b74b7a03c78a', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='fe392124-22dc-406d-a1cb-da95babe86e9', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='a29fa53d-b57a-47d2-a544-780df6879c83', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia')],
 'context': 'Recurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia',
 'patient_id': None,
 'patient_context': 'No patient data was provide

In [13]:
print(result1["answer"])
print(result1["multimodal_result"])

 Based on the provided context, there are no signs or symptoms of breast cancer detected in this case. The multimodal analysis indicates a medium risk level with no critical alert. The identified symptoms include movement observed, initial visual analysis, fear, fatigue, and difficulty sleeping. These symptoms could be associated with various conditions such as stress, anxiety, or other non-cancerous health issues. However, without specific patient data, it is not possible to definitively assess the possibility of breast cancer. It's recommended that the patient follows regular check-ups with their healthcare team for thorough evaluation and appropriate care.
{'final_score': 0.67, 'risk_level': 'medio', 'alert': False, 'evidences': ['movimento corporal identificado', 'análise visual inicial executada', 'medo', 'cansaço', 'dificuldade para dormir'], 'video_score': 0.55, 'audio_score': 0.75}


In [14]:
print(result1["multimodal_context"])

MULTIMODAL ANALYSIS:
Video risk score: 0.55
Audio risk score: 0.75
Final multimodal score: 0.67
Risk level: medio
Alert generated: False

Evidence:
- movimento corporal identificado
- análise visual inicial executada
- medo
- cansaço
- dificuldade para dormir

Alert message:
Sem alerta crítico no momento. Recomenda-se acompanhamento regular pela equipe de saúde.


## Análise do Resultado – Limitações da Abordagem Multimodal

Neste experimento, foi realizada uma pergunta sobre possíveis sinais de câncer de mama, enquanto o pipeline multimodal utilizou dados comportamentais e emocionais simulados (vídeo e áudio).

O sistema respondeu corretamente ao indicar que não é possível identificar câncer de mama com base apenas em sinais comportamentais ou emocionais, reforçando a necessidade de exames clínicos e dados médicos específicos.

A análise multimodal contribuiu apenas com um score de risco geral (médio), baseado em sinais como medo, cansaço e dificuldade para dormir, que são inespecíficos e podem estar associados a diversas condições não relacionadas ao câncer.

Esse resultado evidencia uma limitação importante da abordagem:
dados multimodais como áudio e vídeo são úteis para triagem e identificação de bem-estar psicológico, mas não são suficientes para diagnóstico de doenças específicas como câncer.

O sistema manteve comportamento seguro, evitando conclusões médicas indevidas e recomendando acompanhamento profissional, alinhado aos princípios de segurança definidos na arquitetura.

In [5]:
import speech_recognition as sr

print("OK")

OK


In [7]:
import speech_recognition as sr

def transcribe_audio(audio_path):
    recognizer = sr.Recognizer()

    with sr.AudioFile(audio_path) as source:
        audio = recognizer.record(source)

    text = recognizer.recognize_google(audio, language="pt-BR")

    return text


audio_path = "../data/multimodal/audio/audio1.wav"

transcription = transcribe_audio(audio_path)
print(transcription)

audio_path2 = "../data/multimodal/audio/audio2.wav"

transcription2 = transcribe_audio(audio_path2)
print(transcription2)

audio_path3 = "../data/multimodal/audio/audio3.wav"

transcription3 = transcribe_audio(audio_path3)
print(transcription3)

Oi boa tarde venho apenas para uma consulta de rotina e gostaria de fazer um check-up geral para avaliar como está minha saúde
eu tenho me sentindo muito cansada ultimamente um pouco ansiosa fico um pouco agitada também às vezes eu esqueço das coisas muito rápido
eu tenho me sentido muito mal eu acordo fico com vontade de ficar na cama o dia inteiro eu não consigo dormir fico com uma sensação de angústia medo ao longo do dia não tenho vontade de fazer nada


In [9]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

Base vetorial carregada de: ../data/vector_store


In [10]:
state = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/audio1.wav"
}

In [11]:
result = app.invoke(state)

print(result["audio_result"]["transcription"])
print(result["audio_result"]["flags"])
print(result["multimodal_result"])

Oi boa tarde venho apenas para uma consulta de rotina e gostaria de fazer um check-up geral para avaliar como está minha saúde
[]
{'final_score': 0.43, 'risk_level': 'medio', 'alert': False, 'evidences': ['movimento corporal identificado', 'análise visual inicial executada'], 'video_score': 0.55, 'audio_score': 0.35}


In [17]:
state1 = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/audio2.wav"
}

result1 = app.invoke(state1)

print(result1["audio_result"]["transcription"])
print(result1["audio_result"]["flags"])
print(result1["multimodal_result"])

eu tenho me sentindo muito cansada ultimamente um pouco ansiosa fico um pouco agitada também às vezes eu esqueço das coisas muito rápido
['cansada', 'ansiosa', 'agitada']
{'final_score': 0.67, 'risk_level': 'medio', 'alert': False, 'evidences': ['movimento corporal identificado', 'análise visual inicial executada', 'cansada', 'ansiosa', 'agitada'], 'video_score': 0.55, 'audio_score': 0.75}


In [18]:
state2 = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/audio3.wav"
}

result2 = app.invoke(state2)

print(result2["audio_result"]["transcription"])
print(result2["audio_result"]["flags"])
print(result2["multimodal_result"])

eu tenho me sentido muito mal eu acordo fico com vontade de ficar na cama o dia inteiro eu não consigo dormir fico com uma sensação de angústia medo ao longo do dia não tenho vontade de fazer nada
['medo', 'angústia', 'não consigo dormir', 'não tenho vontade', 'muito mal']
{'final_score': 0.82, 'risk_level': 'alto', 'alert': True, 'evidences': ['movimento corporal identificado', 'análise visual inicial executada', 'medo', 'angústia', 'não consigo dormir', 'não tenho vontade', 'muito mal'], 'video_score': 0.55, 'audio_score': 1.0}


## Teste com Áudio Real

Nesta etapa, o pipeline foi atualizado para utilizar um arquivo de áudio real. A transcrição automática foi executada com sucesso e integrada ao fluxo multimodal do LangGraph.

O resultado demonstra que a arquitetura já é capaz de substituir dados simulados por dados reais sem alterar o restante do pipeline. Nesta execução, a análise de áudio ainda depende de palavras-chave, o que pode limitar a detecção de risco caso a transcrição não contenha exatamente os termos esperados.

Como próximo passo, o módulo de análise textual será refinado com mais variações linguísticas e, futuramente, poderá ser integrado a modelos de linguagem para interpretação contextual mais robusta.

In [6]:
import speech_recognition as sr

print("SpeechRecognition instalado!")

SpeechRecognition instalado!


In [8]:
from src.multimodal.audio_processor import analyze_audio

audio_result = analyze_audio("../data/multimodal/audio/audio3.wav")

print(audio_result["transcription"])
print(audio_result["detected_categories"])
print(audio_result["voice_features"])
print(audio_result["interpretation"])
print(audio_result["recommendation"])

eu tenho me sentido muito mal eu acordo fico com vontade de ficar na cama o dia inteiro eu não consigo dormir fico com uma sensação de angústia medo ao longo do dia não tenho vontade de fazer nada
{'ansiedade': ['angústia', 'medo'], 'depressao_pos_parto_ou_sofrimento_emocional': ['não tenho vontade', 'muito mal'], 'sinais_de_violencia_ou_medo': ['medo'], 'alteracao_do_sono': ['não consigo dormir']}
{'duration_seconds': 26.91, 'rms_energy': 2269, 'voice_intensity': 'alta'}
A análise identificou categorias associadas a ansiedade, depressao_pos_parto_ou_sofrimento_emocional, sinais_de_violencia_ou_medo, alteracao_do_sono. Os principais indicadores encontrados foram: angústia, medo, não tenho vontade, muito mal, não consigo dormir. Esses sinais não representam diagnóstico, mas podem indicar necessidade de atenção especializada.
Recomenda-se avaliação prioritária pela equipe especializada.


## Interpretação da Análise de Áudio com Dados Reais

Nesta etapa, foi realizada a análise de um áudio real contendo relatos de desconforto emocional e possíveis alterações comportamentais.

A transcrição automática identificou expressões relevantes como “angústia”, “medo”, “não tenho vontade” e “não consigo dormir”, permitindo a classificação em múltiplas categorias emocionais, incluindo:

- ansiedade  
- sofrimento emocional (possível depressão pós-parto)  
- sinais de medo ou vulnerabilidade  
- alteração do sono  

Além da análise textual, foram extraídas características do sinal de áudio, como duração e intensidade vocal. A intensidade foi classificada como **alta**, o que pode indicar maior carga emocional durante a fala.

Com base na combinação desses fatores, o sistema atribuiu um nível de risco elevado, recomendando avaliação prioritária por equipe especializada.

É importante destacar que os resultados não representam um diagnóstico clínico, mas sim um mecanismo de triagem inicial baseado em padrões linguísticos e comportamentais. A abordagem multimodal permite identificar sinais precoces de sofrimento psicológico, contribuindo para intervenções mais rápidas e direcionadas.

In [7]:
import os
print(os.getcwd())

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks


In [8]:
import os

print(os.path.exists(".env"))

True


In [9]:
import sys
!{sys.executable} -m pip install boto3 python-dotenv


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Integração com AWS S3 – Teste de Conectividade

Na célula abaixo foi realizada a validação da integração do sistema com o serviço de armazenamento em nuvem Amazon S3, garantindo que o pipeline seja capaz de:

- Realizar upload de arquivos
- Realizar download de arquivos
- Remover arquivos após processamento

Essa etapa é essencial para suportar a arquitetura multimodal do projeto, permitindo o armazenamento seguro de áudios e vídeos.

---

### Configuração

Foram utilizadas as seguintes tecnologias:

- AWS S3 para armazenamento em nuvem
- boto3 como SDK Python para integração com a AWS
- python-dotenv para gerenciamento de credenciais

As credenciais foram armazenadas em um arquivo `.env`:

```env
AWS_ACCESS_KEY_ID=***
AWS_SECRET_ACCESS_KEY=***
AWS_REGION=us-east-1
AWS_S3_BUCKET=nome-do-bucket

In [10]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

s3 = boto3.client("s3", region_name=region)

# cria pasta temp
os.makedirs("temp", exist_ok=True)

# arquivo local
with open("temp/teste.txt", "w") as f:
    f.write("Teste S3 FIAP")

# upload
s3.upload_file("temp/teste.txt", bucket, "temp/teste.txt")
print("Upload OK")

# download
s3.download_file(bucket, "temp/teste.txt", "temp/teste_baixado.txt")
print("Download OK")

# leitura
with open("temp/teste_baixado.txt") as f:
    print("Conteúdo:", f.read())

# delete
s3.delete_object(Bucket=bucket, Key="temp/teste.txt")
print("Delete OK")

Upload OK
Download OK
Conteúdo: Teste S3 FIAP
Delete OK


## Upload de áudio no AWS S3

Áudio real importado no bucket criado no AWS S3.

In [11]:
audio_local = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\audio2.wav"

s3.upload_file(
    audio_local,
    bucket,
    "inputs/audios/audio_real.wav"
)

print("Upload realizado com sucesso")

Upload realizado com sucesso


In [12]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks\..\src\llm\ollama_client.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


Base vetorial carregada de: ../data/vector_store


In [23]:
state = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "audio_s3_key": "inputs/audio/audio_real.wav",
    "video_s3_key": None,
    "patient_id": None
}

result = app.invoke(state)

print(result["audio_result"]["transcription"])
print(result["audio_result"]["flags"])
print(result["multimodal_result"])

eu tenho me sentindo muito cansada ultimamente um pouco ansiosa fico um pouco agitada também às vezes eu esqueço das coisas muito rápido
['ansiosa', 'agitada', 'cansada']
{'final_score': 0.39, 'risk_level': 'baixo', 'alert': False, 'evidences': ['ansiosa', 'agitada', 'cansada'], 'video_score': 0, 'audio_score': 0.65}


## Interpretação do Score Multimodal

### Contexto do Teste

Foi realizado um teste com entrada multimodal contendo:

- Áudio real da paciente (processado via S3)
- Ausência de vídeo (`video_s3_key = None`)
- Pergunta neutra focada em triagem clínica

---

### Resultado Obtido

```json
{
  "final_score": 0.39,
  "risk_level": "baixo",
  "alert": false,
  "evidences": ["ansiosa", "agitada", "cansada"],
  "video_score": 0,
  "audio_score": 0.65
}



### Observações

Apesar do audio_score indicar um nível moderado de risco (0.65), o final_score foi reduzido para 0.39 devido à ausência de dados de vídeo.

Isso evidencia que:

- O sistema considera múltiplas fontes de evidência
- A ausência de uma modalidade pode impactar o resultado final
- A análise multimodal é mais robusta quando ambas as entradas estão disponíveis

Limitações:
- A ausência de vídeo reduz a capacidade de avaliação comportamental
- O score final pode subestimar o risco quando apenas uma modalidade é utilizada
- O sistema não realiza diagnóstico, apenas apoio à triagem

In [33]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

s3 = boto3.client("s3", region_name=region)

# caminho do arquivo no S3 (igual você usou no upload)
s3_key = "inputs/audio/audio_real.wav"

s3.delete_object(Bucket=bucket, Key=s3_key)

print("Arquivo removido do S3 com sucesso")

Arquivo removido do S3 com sucesso


## Etapa de Análise de Vídeo (Multimodal)

### Objetivo

Implementar a análise de vídeo como parte do pipeline multimodal, permitindo a extração de informações visuais que possam complementar a avaliação de risco baseada em áudio e contexto clínico.

Esta etapa tem como finalidade incorporar sinais comportamentais e contextuais, contribuindo para uma análise mais completa, sem caráter diagnóstico.

---

### Tecnologias Utilizadas

- OpenCV: leitura e manipulação de vídeo
- YOLOv8 (Ultralytics): detecção de objetos em frames
- Python: implementação da lógica de análise

---

### Fluxo de Processamento

```text
Vídeo (.mp4)
→ leitura com OpenCV
→ extração de frames
→ detecção com YOLOv8
→ identificação de elementos relevantes (ex: pessoa)
→ geração de métricas e sinais
→ output estruturado

In [12]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import cv2

In [14]:
import os

video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"

print(os.path.exists(video_path))

True


In [15]:
import cv2
import os

video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"
print("Existe?", os.path.exists(video_path))
print("Tamanho bytes:", os.path.getsize(video_path))

cap = cv2.VideoCapture(video_path)

print("Abriu?", cap.isOpened())
print("Frames informados:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Largura:", cap.get(cv2.CAP_PROP_FRAME_WIDTH))
print("Altura:", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

ret, frame = cap.read()

print("Conseguiu ler primeiro frame?", ret)
print("Frame:", None if frame is None else frame.shape)

cap.release()

Existe? True
Tamanho bytes: 38292439
Abriu? True
Frames informados: 3326.0
FPS: 30.0
Largura: 1280.0
Altura: 720.0
Conseguiu ler primeiro frame? True
Frame: (720, 1280, 3)


### Validação da Leitura de Vídeo com OpenCV

Validar a capacidade do sistema de:

- localizar o arquivo de vídeo
- abrir o vídeo corretamente
- acessar metadados do conteúdo
- ler frames reais para processamento posterior

Essa etapa é necessária antes da integração com o modelo YOLOv8 e com o pipeline multimodal.


In [15]:
!pip install ultralytics


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import cv2
from ultralytics import YOLO

# carrega modelo
model = YOLO("yolov8n.pt")

video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

if ret:
    results = model(frame)

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            label = model.names[cls]

            print("Detectado:", label)

cap.release()


0: 384x640 4 persons, 290.2ms
Speed: 61.2ms preprocess, 290.2ms inference, 33.5ms postprocess per image at shape (1, 3, 384, 640)
Detectado: person
Detectado: person
Detectado: person
Detectado: person


## Teste Inicial com YOLOv8 em Frame de Vídeo

### Objetivo

Validar o uso do modelo YOLOv8 para detecção de objetos em frames extraídos do vídeo real utilizado no projeto.

Nesta etapa, o objetivo foi verificar se o modelo consegue identificar elementos visuais relevantes no vídeo antes da integração completa com o pipeline multimodal.

---

### Procedimento

Foi carregado o modelo pré-treinado `yolov8n.pt` e extraído o primeiro frame do vídeo. Em seguida, o frame foi enviado ao modelo YOLO para detecção.

---

### Resultado Obtido

O modelo identificou múltiplas ocorrências da classe:

```text
person

## Análise Emocional Facial com AWS Rekognition

### Objetivo

Complementar a análise multimodal do sistema com identificação de emoções faciais em vídeos, utilizando um serviço gerenciado em nuvem.

Nesta etapa, foi utilizada a API Amazon Rekognition para detectar faces e inferir emoções aparentes a partir de frames extraídos do vídeo.

---

### Serviço em Nuvem Utilizado

- AWS Rekognition
- AWS S3 (armazenamento dos arquivos multimodais)

O Rekognition foi integrado como serviço cognitivo complementar, permitindo realizar análise facial sem necessidade de treinamento local de modelos complexos.

---

### Fluxo da Análise

```text
- vídeo (.mp4)
- leitura com OpenCV
- extração de frames periódicos
- envio dos frames para AWS Rekognition
- detecção facial
- identificação de emoções
- geração de score visual
- integração multimodal

In [16]:
import sys
!{sys.executable} -m pip install boto3 opencv-python python-dotenv


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

region = os.getenv("AWS_REGION", "us-east-1")

rekognition = boto3.client(
    "rekognition",
    region_name=region
)

print("Cliente Rekognition criado")

Cliente Rekognition criado


In [22]:
import cv2

video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

if not ret:
    raise Exception("Não foi possível ler o primeiro frame.")

success, encoded_image = cv2.imencode(".jpg", frame)

if not success:
    raise Exception("Não foi possível converter o frame para JPG.")

response = rekognition.detect_faces(
    Image={"Bytes": encoded_image.tobytes()},
    Attributes=["ALL"]
)

response["FaceDetails"]

[{'BoundingBox': {'Width': 0.1466677188873291,
   'Height': 0.4765356183052063,
   'Left': 0.03128409385681152,
   'Top': 0.21017633378505707},
  'AgeRange': {'Low': 22, 'High': 28},
  'Smile': {'Value': False, 'Confidence': 99.09146881103516},
  'Eyeglasses': {'Value': True, 'Confidence': 99.98126220703125},
  'Sunglasses': {'Value': False, 'Confidence': 99.48003387451172},
  'Gender': {'Value': 'Female', 'Confidence': 99.83406066894531},
  'Beard': {'Value': False, 'Confidence': 98.07129669189453},
  'Mustache': {'Value': False, 'Confidence': 99.70878601074219},
  'EyesOpen': {'Value': True, 'Confidence': 99.99838256835938},
  'MouthOpen': {'Value': False, 'Confidence': 93.41654205322266},
  'Emotions': [{'Type': 'CALM', 'Confidence': 99.72098541259766},
   {'Type': 'HAPPY', 'Confidence': 0.037034355103969574},
   {'Type': 'SURPRISED', 'Confidence': 0.02384185791015625},
   {'Type': 'CONFUSED', 'Confidence': 0.02155701443552971},
   {'Type': 'ANGRY', 'Confidence': 0.01878738403320312

In [23]:
faces = response.get("FaceDetails", [])

if not faces:
    print("Nenhuma face detectada.")
else:
    emotions = faces[0].get("Emotions", [])
    dominant_emotion = max(emotions, key=lambda e: e["Confidence"])

    print("Emoção dominante:", dominant_emotion["Type"])
    print("Confiança:", round(dominant_emotion["Confidence"], 2))

Emoção dominante: CALM
Confiança: 99.72


In [24]:
import cv2
from collections import Counter

video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"

cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
step = fps * 5

frame_id = 0
frames_analyzed = 0
emotions_detected = []
faces_detected = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_id % step == 0:
        success, encoded_image = cv2.imencode(".jpg", frame)

        if success:
            response = rekognition.detect_faces(
                Image={"Bytes": encoded_image.tobytes()},
                Attributes=["ALL"]
            )

            faces = response.get("FaceDetails", [])

            if faces:
                faces_detected += len(faces)

                for face in faces:
                    emotions = face.get("Emotions", [])

                    if emotions:
                        dominant = max(emotions, key=lambda e: e["Confidence"])
                        emotions_detected.append(dominant["Type"])

            frames_analyzed += 1

    frame_id += 1

cap.release()

print("Frames analisados:", frames_analyzed)
print("Faces detectadas:", faces_detected)
print("Emoções detectadas:", emotions_detected)
print("Contagem:", Counter(emotions_detected))

Frames analisados: 23
Faces detectadas: 33
Emoções detectadas: ['CALM', 'CALM', 'CALM', 'CALM', 'CALM', 'CALM', 'CALM', 'HAPPY', 'CALM', 'SURPRISED', 'FEAR', 'CALM', 'ANGRY', 'HAPPY', 'CONFUSED', 'CALM', 'CONFUSED', 'CALM', 'CALM', 'CALM', 'SAD', 'CALM', 'CALM', 'CALM', 'SURPRISED', 'CALM', 'SAD', 'SAD', 'SURPRISED', 'HAPPY', 'HAPPY', 'HAPPY', 'CALM']
Contagem: Counter({'CALM': 18, 'HAPPY': 5, 'SURPRISED': 3, 'SAD': 3, 'CONFUSED': 2, 'FEAR': 1, 'ANGRY': 1})


In [25]:
emotion_weights = {
    "FEAR": 0.8,
    "SAD": 0.6,
    "ANGRY": 0.5,
    "DISGUSTED": 0.4,
    "SURPRISED": 0.3,
    "CONFUSED": 0.3,
    "CALM": 0.2,
    "UNKNOWN": 0.2,
    "HAPPY": 0.0
}

if emotions_detected:
    scores = [emotion_weights.get(e, 0.2) for e in emotions_detected]
    video_score = round(sum(scores) / len(scores), 2)
else:
    video_score = 0

if video_score >= 0.75:
    risk_level = "alto"
elif video_score >= 0.45:
    risk_level = "medio"
elif video_score > 0:
    risk_level = "baixo"
else:
    risk_level = "not_provided"

flags = []

emotion_count = Counter(emotions_detected)

if faces_detected > 0:
    flags.append("face_detected")

for emotion in ["FEAR", "SAD", "ANGRY", "DISGUSTED", "CONFUSED"]:
    if emotion_count.get(emotion, 0) > 0:
        flags.append(f"{emotion.lower()}_expression")

video_result = {
    "modality": "video",
    "risk_score": video_score,
    "risk_level": risk_level,
    "flags": flags,
    "metadata": {
        "frames_analyzed": frames_analyzed,
        "faces_detected": faces_detected,
        "dominant_emotions": dict(emotion_count)
    }
}

video_result

{'modality': 'video',
 'risk_score': 0.25,
 'risk_level': 'baixo',
 'flags': ['face_detected',
  'fear_expression',
  'sad_expression',
  'angry_expression',
  'confused_expression'],
 'metadata': {'frames_analyzed': 23,
  'faces_detected': 33,
  'dominant_emotions': {'CALM': 18,
   'HAPPY': 5,
   'SURPRISED': 3,
   'FEAR': 1,
   'ANGRY': 1,
   'CONFUSED': 2,
   'SAD': 3}}}

### Resultado da Análise Emocional Facial

A integração com o AWS Rekognition foi realizada com sucesso, permitindo detectar emoções faciais aparentes a partir de frames extraídos do vídeo.

### Resultado Obtido

```json
{
  "risk_score": 0.25,
  "risk_level": "baixo",
  "flags": [
    "face_detected",
    "fear_expression",
    "sad_expression",
    "angry_expression",
    "confused_expression"
  ]
}

## Teste Completo do Pipeline Multimodal com LangGraph

Nesta etapa foi realizado o teste completo do pipeline multimodal utilizando o LangGraph como orquestrador principal do fluxo de IA.

O objetivo foi validar a integração entre:

- armazenamento em nuvem com AWS S3;
- processamento de vídeo;
- processamento de áudio;
- fusão multimodal;
- geração de resposta interpretativa com LLM.

O fluxo executado foi:

entrada do usuário
→ cloud storage
→ análise de vídeo
→ análise de áudio
→ fusão multimodal
→ geração de resposta
→ alerta e logging

O teste utilizou:
- um vídeo real armazenado no Amazon S3;
- um áudio real armazenado no Amazon S3;
- processamento multimodal integrado;
- geração automática de interpretação clínica educacional.

O sistema não realiza diagnóstico médico.
A análise possui finalidade exclusivamente educacional e de apoio à triagem clínica preventiva.

In [13]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

s3 = boto3.client("s3", region_name=region)

local_video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"
video_s3_key = "inputs/videos/video_real.mp4"

s3.upload_file(
    local_video_path,
    bucket,
    video_s3_key
)

print("Upload concluído:", video_s3_key)

Upload concluído: inputs/videos/video_real.mp4


In [20]:
os.makedirs("temp", exist_ok=True)

local_temp_video = "temp/video_real.mp4"

s3.download_file(
    bucket,
    video_s3_key,
    local_temp_video
)

print("Download concluído:", local_temp_video)
print("Existe localmente?", os.path.exists(local_temp_video))

Download concluído: temp/video_real.mp4
Existe localmente? True


In [22]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks\..\src\llm\ollama_client.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


Base vetorial carregada de: ../data/vector_store


### Execução do Pipeline Multimodal

Nesta etapa o pipeline completo foi executado através do método:

app.invoke(state)

O LangGraph coordenou automaticamente:

1. Download dos arquivos do S3
2. Processamento de vídeo
3. Processamento de áudio
4. Fusão multimodal
5. Geração da resposta com LLM
6. Registro da interação

Essa execução valida o funcionamento integrado da arquitetura multimodal proposta para o projeto.

In [27]:
state = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "audio_s3_key": "inputs/audio/audio_real.wav",
    "video_s3_key": "inputs/videos/video_real.mp4",
    "patient_id": None
}

result = app.invoke(state)

In [28]:
result["video_result"]

{'modality': 'video',
 'risk_score': 0.25,
 'risk_level': 'baixo',
 'flags': ['person_detected',
  'face_detected',
  'fear_expression',
  'sad_expression',
  'angry_expression',
  'confused_expression'],
 'metadata': {'video_path': 'temp\\video_real.mp4',
  'fps': 30.0,
  'frame_count': 3326,
  'duration_seconds': 110.87,
  'frames_analyzed': 23,
  'faces_detected': 33,
  'dominant_emotions': {'CALM': 18,
   'HAPPY': 5,
   'SURPRISED': 3,
   'FEAR': 1,
   'ANGRY': 1,
   'CONFUSED': 2,
   'SAD': 3},
  'cloud_service': 'AWS Rekognition',
  'visual_model': 'YOLOv8'}}

### Resultado da Análise de Vídeo

Nesta etapa foi exibido o resultado do processamento visual do vídeo.

Tecnologias utilizadas:
- OpenCV para leitura do vídeo;
- YOLOv8 para detecção de presença humana;
- AWS Rekognition para análise facial e emocional.

O sistema realizou:
- extração periódica de frames;
- detecção de rostos;
- identificação de emoções aparentes;
- cálculo de score de risco visual.

Resultado observado:
- presença humana detectada;
- múltiplas faces identificadas;
- emoções como fear, sad, angry e confused foram detectadas;
- emoção predominante: calm.

Interpretação:
Embora tenham sido observados sinais complementares de desconforto emocional, a predominância da emoção CALM reduziu o score final de risco visual.

Limitações:
A análise facial detecta apenas expressões aparentes e não representa estado psicológico real.

In [29]:
result["audio_result"]

{'modality': 'audio',
 'risk_score': 0.65,
 'risk_level': 'medio',
 'flags': ['ansiosa', 'agitada', 'cansada'],
 'detected_categories': {'ansiedade': ['ansiosa', 'agitada'],
  'fadiga_hormonal_ou_cansaco': ['cansada']},
 'voice_features': {'duration_seconds': 15.51,
  'rms_energy': 2006,
  'voice_intensity': 'alta'},
 'transcription': 'eu tenho me sentindo muito cansada ultimamente um pouco ansiosa fico um pouco agitada também às vezes eu esqueço das coisas muito rápido',
 'interpretation': 'A análise identificou categorias associadas a ansiedade, fadiga_hormonal_ou_cansaco. Os principais indicadores encontrados foram: ansiosa, agitada, cansada. Esses sinais não representam diagnóstico, mas podem indicar necessidade de atenção especializada.',
 'recommendation': 'Recomenda-se acompanhamento e nova avaliação clínica.'}

### Resultado da Análise de Áudio

Nesta etapa foi exibido o resultado do processamento de áudio da consulta.

O sistema realizou:
- transcrição automática;
- análise emocional da fala;
- detecção de palavras-chave emocionais;
- classificação de risco.

Indicadores detectados:
- ansiedade;
- agitação;
- cansaço.

Resultado observado:
- score de risco médio;
- intensidade vocal elevada;
- presença de termos associados a ansiedade e fadiga emocional.

Interpretação:
O áudio apresentou sinais compatíveis com possível desconforto emocional e necessidade de atenção clínica complementar.

Limitações:
O sistema não realiza diagnóstico psicológico ou psiquiátrico.
Os resultados representam apenas apoio à triagem preventiva.

In [30]:
result["multimodal_result"]

{'final_score': 0.49,
 'risk_level': 'medio',
 'alert': False,
 'evidences': ['person_detected',
  'face_detected',
  'fear_expression',
  'sad_expression',
  'angry_expression',
  'confused_expression',
  'ansiosa',
  'agitada',
  'cansada'],
 'video_score': 0.25,
 'audio_score': 0.65}

### Resultado da Fusão Multimodal

Nesta etapa foi realizada a fusão entre os resultados de áudio e vídeo.

O objetivo da fusão multimodal é combinar diferentes evidências emocionais e comportamentais para gerar uma avaliação mais robusta.

O sistema combinou:
- score visual;
- score de áudio;
- evidências emocionais detectadas.

Resultado observado:
- score multimodal final: médio;
- evidências combinadas de ansiedade, medo, tristeza e agitação;
- ausência de alerta crítico.

Interpretação:
A combinação multimodal aumentou a robustez da análise, permitindo integrar sinais visuais e sonoros associados a possível desconforto psicológico.

Essa abordagem reduz a dependência de apenas uma modalidade de entrada.

In [31]:
print(result["answer"])

 Based on the provided structured data and multimodal analysis, the following signs of potential risk were detected in the patient's presentation: fear, sadness, anger, confusion, anxiety, agitation, and fatigue. The multimodal score indicates a medium level of risk, and no critical alert was generated. Therefore, it is recommended that the patient should have regular follow-up with her healthcare team. However, since there's no specific patient data provided, further evaluation by a healthcare professional is necessary to confirm these findings and determine any appropriate interventions or referrals for psychological support if needed.


### Resposta Final Gerada pelo LLM

Nesta etapa foi exibida a resposta final produzida pelo modelo Mistral.

O LLM utilizou:
- contexto multimodal;
- informações estruturadas;
- dados recuperados pelo sistema RAG.

A resposta foi gerada em linguagem educacional e interpretativa, seguindo restrições de segurança clínica.

Características da resposta:
- não fornece diagnóstico;
- não prescreve tratamento;
- apresenta interpretação cautelosa;
- recomenda avaliação profissional quando necessário.

O objetivo é fornecer apoio à triagem clínica preventiva em saúde feminina.

In [51]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

s3 = boto3.client(
    "s3",
    region_name=region
)

video_s3_key = "inputs/videos/video_real.mp4"

# remover objeto do S3
s3.delete_object(
    Bucket=bucket,
    Key=video_s3_key
)

print(f"Vídeo removido do S3: {video_s3_key}")

Vídeo removido do S3: inputs/videos/video_real.mp4


## TESTES APÓS MELHORIAS NO LLM

### Teste da Análise de Áudio

Este notebook valida a função `analyze_audio()` após os aprimoramentos de análise vocal.

O objetivo é verificar:
- transcrição do áudio;
- identificação de palavras-chave emocionais;
- extração de características acústicas;
- análise de tom de voz;
- identificação de hesitação, tensão vocal, baixa energia ou instabilidade;
- cálculo do score de risco de áudio.

A análise é apenas apoio à triagem e não representa diagnóstico médico, psicológico ou psiquiátrico.

In [15]:
import os
from pprint import pprint

from src.multimodal.audio_processor import analyze_audio

In [19]:
audio_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\audio3.wav"

print("Áudio existe?", os.path.exists(audio_path))
print("Caminho:", audio_path)



Áudio existe? True
Caminho: C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\audio3.wav


In [20]:
audio_result = analyze_audio(audio_path)

pprint(audio_result)

{'detected_categories': {'alteracao_do_sono': ['não consigo dormir'],
                         'ansiedade': ['angústia', 'medo'],
                         'depressao_pos_parto_ou_sofrimento_emocional': ['não '
                                                                         'tenho '
                                                                         'vontade',
                                                                         'muito '
                                                                         'mal'],
                         'sinais_de_violencia_ou_medo': ['medo', 'dor']},
 'flags': ['angústia',
           'medo',
           'não tenho vontade',
           'muito mal',
           'dor',
           'não consigo dormir',
           'voice_instability',
           'elevated_voice_tension',
           'speech_hesitation'],
 'interpretation': 'A análise identificou categorias associadas a ansiedade, '
                   'depressao_pos_parto_ou_sofrimento_emo

In [21]:
print("Score de risco:", audio_result.get("risk_score"))
print("Nível de risco:", audio_result.get("risk_level"))
print("Flags:", audio_result.get("flags"))

Score de risco: 1.0
Nível de risco: alto
Flags: ['angústia', 'medo', 'não tenho vontade', 'muito mal', 'dor', 'não consigo dormir', 'voice_instability', 'elevated_voice_tension', 'speech_hesitation']


In [22]:
print(audio_result.get("transcription"))

eu tenho me sentido muito mal eu acordo fico com vontade de ficar na cama o dia inteiro eu não consigo dormir fico com uma sensação de angústia medo ao longo do dia não tenho vontade de fazer nada


In [23]:
pprint(audio_result.get("detected_categories"))

{'alteracao_do_sono': ['não consigo dormir'],
 'ansiedade': ['angústia', 'medo'],
 'depressao_pos_parto_ou_sofrimento_emocional': ['não tenho vontade',
                                                 'muito mal'],
 'sinais_de_violencia_ou_medo': ['medo', 'dor']}


In [25]:
voice_features = audio_result.get("voice_features", {})

pprint(voice_features)

{'duration_seconds': 26.91,
 'mean_energy': 0.0473,
 'mean_pitch': 1252.05,
 'pitch_variation': 1070.85,
 'silence_ratio': 0.53,
 'tone_flags': ['voice_instability',
                'elevated_voice_tension',
                'speech_hesitation'],
 'voice_intensity': 'moderada'}


In [26]:
tone_flags = voice_features.get("tone_flags", [])

print("Sinais vocais detectados:")

if tone_flags:
    for flag in tone_flags:
        print("-", flag)
else:
    print("- Nenhum sinal vocal relevante identificado.")

Sinais vocais detectados:
- voice_instability
- elevated_voice_tension
- speech_hesitation


In [27]:
print("Interpretação:")
print(audio_result.get("interpretation"))

print("\nRecomendação:")
print(audio_result.get("recommendation"))

Interpretação:
A análise identificou categorias associadas a ansiedade, depressao_pos_parto_ou_sofrimento_emocional, sinais_de_violencia_ou_medo, alteracao_do_sono. Os principais indicadores encontrados foram: angústia, medo, não tenho vontade, muito mal, dor, não consigo dormir, voice_instability, elevated_voice_tension, speech_hesitation. Esses sinais não representam diagnóstico, mas podem indicar necessidade de atenção especializada.

Recomendação:
Recomenda-se avaliação prioritária pela equipe especializada.


### Interpretação do resultado de áudio

A análise de áudio combina duas abordagens:

1. Análise textual da transcrição:
   - ansiedade;
   - medo;
   - fadiga;
   - sofrimento emocional;
   - possíveis sinais de violência ou insegurança.

2. Análise acústica da voz:
   - energia vocal;
   - pitch médio;
   - variação do pitch;
   - proporção de silêncio;
   - sinais de hesitação;
   - tensão vocal;
   - baixa energia;
   - instabilidade vocal.

Essas características permitem avaliar não apenas o conteúdo falado, mas também aspectos do tom de voz.

A análise não representa diagnóstico. Os sinais identificados são evidências complementares para triagem clínica preventiva.

## Teste da Análise de Vídeo

Este notebook valida a função `analyze_video()` após os aprimoramentos de interpretação emocional visual.

O objetivo é verificar:
- leitura do vídeo;
- extração de metadados com OpenCV;
- detecção de pessoa com YOLOv8;
- análise facial com AWS Rekognition;
- persistência emocional;
- variação emocional;
- score de risco visual;
- interpretação automática.

A análise é apenas apoio à triagem e não representa diagnóstico.

In [30]:
import os
from dotenv import load_dotenv
from pprint import pprint

from src.multimodal.video_processor import (
    extract_video_metadata,
    analyze_video
)

load_dotenv()

True

In [31]:
print("AWS_REGION:", os.getenv("AWS_REGION"))
print("AWS_S3_BUCKET:", os.getenv("AWS_S3_BUCKET"))

AWS_REGION: us-east-1
AWS_S3_BUCKET: postechfase4


In [32]:
video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"

print("Vídeo existe?", os.path.exists(video_path))
print("Caminho:", video_path)

Vídeo existe? True
Caminho: C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4


In [33]:
metadata = extract_video_metadata(video_path)

pprint(metadata)

{'duration_seconds': 110.87,
 'fps': 30.0,
 'frame_count': 3326,
 'video_path': 'C:\\Users\\keity\\OneDrive\\Documentos\\GitHub\\postech\\tech-challenge-ia-diagnostico\\tech-challenge-ia-diagnostico\\data\\multimodal\\video\\Unlocking '
               'Facial Recognition.mp4'}


### Interpretação dos metadados

Nesta etapa, o vídeo foi aberto com OpenCV para validar se está apto para processamento.

São verificados:
- FPS;
- quantidade de frames;
- duração aproximada;
- caminho do arquivo.

Se o vídeo não abrir corretamente, a análise emocional não deve ser executada.

In [35]:
video_result = analyze_video(video_path)

pprint(video_result)

{'flags': ['person_detected', 'face_not_visible', 'multiple_people_detected'],
 'interpretation': ['Foi detectada presença humana, mas não houve detecção '
                    'facial suficiente para análise emocional.',
                    'O vídeo apresentou múltiplas pessoas no mesmo frame, '
                    'indicando maior complexidade contextual da cena.'],
 'limitations': ['A análise facial indica apenas expressões aparentes, não '
                 'estado psicológico real.',
                 'O sistema não realiza diagnóstico médico ou psicológico.',
                 'Os sinais visuais devem ser interpretados apenas como apoio '
                 'à triagem.',
                 'Iluminação, ângulo da câmera, qualidade do vídeo e oclusões '
                 'podem afetar os resultados.',
                 'O YOLOv8 utilizado detecta presença humana, mas não '
                 'substitui avaliação clínica especializada.'],
 'metadata': {'analysis_strategy': 'frame_sampling_every

In [36]:
print("Score de risco:", video_result.get("risk_score"))
print("Nível de risco:", video_result.get("risk_level"))
print("Flags:", video_result.get("flags"))

Score de risco: 0.2
Nível de risco: baixo
Flags: ['person_detected', 'face_not_visible', 'multiple_people_detected']


In [37]:
print("Interpretação visual:")

for item in video_result.get("interpretation", []):
    print("-", item)

Interpretação visual:
- Foi detectada presença humana, mas não houve detecção facial suficiente para análise emocional.
- O vídeo apresentou múltiplas pessoas no mesmo frame, indicando maior complexidade contextual da cena.


In [38]:
metadata = video_result.get("metadata", {})

print("Frames analisados:", metadata.get("frames_analyzed"))
print("Faces detectadas:", metadata.get("faces_detected"))
print("Pessoa detectada:", metadata.get("person_detected"))
print("Máximo de pessoas no frame:", metadata.get("max_people_in_frame"))

print("\nEmoções dominantes:")
pprint(metadata.get("dominant_emotions"))

print("\nPercentual de emoções:")
pprint(metadata.get("emotion_percentages"))

print("\nTransições emocionais:", metadata.get("emotion_transitions"))

Frames analisados: 23
Faces detectadas: 0
Pessoa detectada: True
Máximo de pessoas no frame: 5

Emoções dominantes:
{'UNKNOWN': 23}

Percentual de emoções:
{'UNKNOWN': 100.0}

Transições emocionais: 0


### Interpretação do resultado visual

A análise de vídeo combina três etapas principais:

1. OpenCV para leitura e amostragem dos frames.
2. YOLOv8 para identificar presença humana.
3. AWS Rekognition para detectar expressões emocionais aparentes.

Além da emoção dominante, o sistema avalia:
- persistência de medo, tristeza, tensão ou confusão;
- variação emocional ao longo do vídeo;
- ausência de rosto;
- múltiplas pessoas no frame.

Essas informações são usadas como evidências complementares de triagem, especialmente para sinais visuais de desconforto emocional.

A análise não confirma estado psicológico real e não substitui avaliação profissional.

## Teste LangGraph Completo - Após melhorias

In [52]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

s3 = boto3.client("s3", region_name=region)

local_audio_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\audio2.wav"
local_video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"

audio_s3_key = "inputs/audios/audio_real.wav"
video_s3_key = "inputs/video/video_real.mp4"

s3.upload_file(local_audio_path, bucket, audio_s3_key)
s3.upload_file(local_video_path, bucket, video_s3_key)

print("Upload do áudio concluído:", audio_s3_key)
print("Upload do vídeo concluído:", video_s3_key)

Upload do áudio concluído: inputs/audios/audio_real.wav
Upload do vídeo concluído: inputs/video/video_real.mp4


In [44]:
for key in [audio_s3_key, video_s3_key]:
    response = s3.head_object(
        Bucket=bucket,
        Key=key
    )
    print(key, "-", response["ContentLength"], "bytes")

inputs/audio/audio_real.wav - 2736662 bytes
inputs/video/video_real.mp4 - 38292439 bytes


In [43]:
app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever
)

print("LangGraph compilado com sucesso.")

LangGraph compilado com sucesso.


In [45]:
state = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente com base no áudio e no vídeo.",
    "audio_s3_key": audio_s3_key,
    "video_s3_key": video_s3_key,
    "patient_id": None
}

result = app.invoke(state)

print("Pipeline executado com sucesso.")

Pipeline executado com sucesso.


In [46]:
print("Audio path local:", result.get("audio_path"))
print("Video path local:", result.get("video_path"))

Audio path local: temp\audio_real.wav
Video path local: temp\video_real.mp4


In [47]:
print("Existe áudio local?", os.path.exists(result.get("audio_path", "")))
print("Existe vídeo local?", os.path.exists(result.get("video_path", "")))

Existe áudio local? True
Existe vídeo local? True


In [48]:
from pprint import pprint

print("VIDEO RESULT")
pprint(result["video_result"])

print("\nAUDIO RESULT")
pprint(result["audio_result"])

print("\nMULTIMODAL RESULT")
pprint(result["multimodal_result"])

VIDEO RESULT
{'flags': ['person_detected', 'face_not_visible', 'multiple_people_detected'],
 'interpretation': ['Foi detectada presença humana, mas não houve detecção '
                    'facial suficiente para análise emocional.',
                    'O vídeo apresentou múltiplas pessoas no mesmo frame, '
                    'indicando maior complexidade contextual da cena.'],
 'limitations': ['A análise facial indica apenas expressões aparentes, não '
                 'estado psicológico real.',
                 'O sistema não realiza diagnóstico médico ou psicológico.',
                 'Os sinais visuais devem ser interpretados apenas como apoio '
                 'à triagem.',
                 'Iluminação, ângulo da câmera, qualidade do vídeo e oclusões '
                 'podem afetar os resultados.',
                 'O YOLOv8 utilizado detecta presença humana, mas não '
                 'substitui avaliação clínica especializada.'],
 'metadata': {'analysis_strategy': 'frame_s

In [49]:
print(result["answer"])

 1. Resumo da avaliação: A análise multimodal identificou sinais compatíveis com ansiedade, agitação ou fadiga emocional, indicando um nível de risco médio.

2. Evidências observadas: O áudio apresentou sinais de instabilidade vocal, tensão nasal elevada e hesitação na fala, enquanto o vídeo mostrou expressões faciais indicativas de ansiedade, agitação e cansaço.

3. Nível de risco: Médio

4. Recomendação: É recomendado acompanhamento e nova avaliação clínica caso os sinais persistam, se intensifiquem ou estejam associados a sofrimento emocional.

5. Limitações da análise:
   - A análise multimodal é apenas apoio à triagem clínica.
   - O sistema não realiza diagnóstico médico, psicológico ou psiquiátrico.
   - Expressões faciais indicam apenas emoções aparentes.
   - Sinais de áudio e vídeo devem ser interpretados como evidências complementares.
   - A confirmação clínica depende de avaliação profissional.


In [53]:
for key in [audio_s3_key, video_s3_key]:
    s3.delete_object(
        Bucket=bucket,
        Key=key
    )
    print("Removido do S3:", key)

Removido do S3: inputs/audios/audio_real.wav
Removido do S3: inputs/video/video_real.mp4


### Teste Completo do Pipeline Multimodal com LangGraph e AWS S3

Nesta etapa foi realizado o teste completo do pipeline multimodal utilizando o LangGraph como orquestrador principal da aplicação.

O objetivo foi validar o fluxo integrado de:
- upload de arquivos multimodais no Amazon S3;
- download automático pelo pipeline;
- processamento de vídeo;
- processamento de áudio;
- fusão multimodal;
- geração de resposta interpretativa pelo LLM.

### Fluxo executado

O pipeline executado foi:

entrada do usuário
→ recuperação dos arquivos no S3
→ análise de vídeo
→ análise de áudio
→ fusão multimodal
→ geração da resposta
→ alerta preventivo

### Serviços utilizados

- Amazon S3 para armazenamento temporário dos arquivos;
- AWS Rekognition para análise facial e emocional;
- YOLOv8 para detecção de presença humana;
- OpenCV para processamento de vídeo;
- Librosa para análise acústica da voz;
- LangGraph para orquestração do fluxo;
- Mistral via Ollama para geração da resposta final.

### Resultado da análise de vídeo

O vídeo apresentou:
- detecção de presença humana;
- ausência de detecção facial confiável;
- múltiplas pessoas em alguns frames.

Devido às limitações do vídeo analisado, o AWS Rekognition não conseguiu identificar expressões emocionais faciais de forma consistente.

Por esse motivo:
- não foram identificadas emoções dominantes válidas;
- não houve evidências visuais fortes de medo, tristeza ou tensão;
- o score visual permaneceu baixo.

A análise visual foi utilizada apenas como evidência complementar.

### Resultado da análise de áudio

A análise acústica e textual do áudio identificou:
- sinais de ansiedade;
- agitação;
- fadiga emocional;
- instabilidade vocal;
- hesitação na fala.

Além da transcrição textual, foram utilizadas características acústicas como:
- pitch médio;
- variação do pitch;
- intensidade vocal;
- proporção de silêncio.

Esses indicadores contribuíram para aumento do score de risco do áudio.

### Resultado da fusão multimodal

A fusão multimodal combinou:
- score visual;
- score de áudio;
- evidências emocionais;
- sinais vocais.

Como o áudio apresentou mais sinais relevantes do que o vídeo, o score final foi predominantemente influenciado pela modalidade de áudio.

A estratégia de fusão utilizada foi:
- 60% peso para áudio;
- 40% peso para vídeo.

### Observação importante sobre a resposta do agente

Durante o teste, observou-se que o modelo de linguagem gerou uma interpretação textual sugerindo ansiedade também no vídeo.

Entretanto, a análise visual não identificou evidências emocionais fortes no vídeo analisado.

Isso ocorreu porque:
- o LLM interpretou o contexto multimodal de forma mais ampla;
- o áudio apresentou sinais emocionais relevantes;
- o modelo associou parte dessas evidências ao contexto geral da análise.

Como melhoria futura, recomenda-se:
- separar explicitamente as evidências de áudio e vídeo no prompt;
- impedir que o LLM atribua emoções visuais não detectadas;
- tornar a resposta mais fiel às modalidades individuais.

### Limitações identificadas

A análise visual depende fortemente de:
- iluminação;
- qualidade do vídeo;
- visibilidade facial;
- posicionamento da câmera;
- presença de rostos detectáveis.

A ausência de rosto visível reduz significativamente a capacidade de interpretação emocional visual.

## Conclusão

O teste validou com sucesso:
- a integração multimodal;
- o fluxo completo no LangGraph;
- a comunicação com AWS S3;
- a análise de áudio;
- a análise de vídeo;
- a fusão multimodal;
- e a geração automática de respostas clínicas educacionais.

O sistema possui finalidade exclusivamente acadêmica e de apoio à triagem preventiva, não realizando diagnóstico médico ou psicológico.